# Thực hành Bài kiểm tra 1 – Khảo sát dữ liệu ViHSD

> **Lưu ý:** Notebook mặc định chạy bằng fixture nhỏ, an toàn, được tạo riêng cho kiểm thử kỹ thuật. Fixture **không phải dữ liệu ViHSD** và mọi số liệu/biểu đồ ở chế độ này không được đưa vào báo cáo nghiên cứu. Khi đã có quyền truy cập, đổi `USE_SAMPLE_FIXTURE` thành `False`, chỉ đường dẫn CSV và điền đúng tên cột đã xác minh. Không lưu notebook có output chứa bình luận thật lên GitHub.

Notebook cho phép đếm nhãn, vẽ phân bố, thống kê độ dài, kiểm tra trùng, sửa một quy tắc chuẩn hóa và so sánh trước/sau.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt

import pandas as pd

from vihsd.data import read_csv, validate_schema
from vihsd.eda import (
    comment_lengths,
    count_labels,
    length_summary,
    plot_comment_lengths,
    plot_label_distribution,
)
from vihsd.preprocessing import (
    DEFAULT_ABBREVIATIONS,
    compare_text_versions,
    normalize_series,
    preprocessing_summary,
)
from vihsd.quality import quality_issue_table

USE_SAMPLE_FIXTURE = True
CSV_PATH = Path("data/raw/<tep_csv>")
TEXT_COLUMN = ""  # Dien ten cot van ban sau khi xem schema
LABEL_COLUMN = ""  # Dien ten cot nhan sau khi xem schema
OUTPUT_DIR = Path(os.environ.get("VIHSD_NOTEBOOK_OUTPUT_DIR", "artifacts/practice"))
SHOW_TEXT_PREVIEW = False  # Chi bat neu duoc phep xem noi dung mau cuc bo

if USE_SAMPLE_FIXTURE:
    frame = pd.DataFrame(
        {
            "comment": [
                "Xin chao, ban!",
                "Tks ban",
                "ko thich dieu do",
                "noi dung xuc pham gia lap",
                "nhan xet thu ghet gia lap",
                "Moi nguoi ton trong nhau",
            ],
            "label": ["CLEAN", "CLEAN", "OFFENSIVE", "OFFENSIVE", "HATE", "CLEAN"],
        }
    )
    TEXT_COLUMN, LABEL_COLUMN = "comment", "label"
    SOURCE_MODE = "FIXTURE MINH HOA — KHONG PHAI ViHSD"
else:
    if not TEXT_COLUMN or not LABEL_COLUMN:
        raise ValueError("Hay dien ten cot van ban/nhan da xac minh trong CSV")
    frame = read_csv(CSV_PATH)
    validate_schema(frame, TEXT_COLUMN, LABEL_COLUMN)
    SOURCE_MODE = f"CSV cuc bo: {CSV_PATH}"

print(SOURCE_MODE)
print(f"So dong: {len(frame)} | So cot: {frame.shape[1]}")
print(frame.dtypes.to_string())

## 1. Đếm mẫu theo nhãn và vẽ lại biểu đồ

Đọc bảng và nêu lớp nhiều mẫu nhất. Nếu dùng fixture, chỉ mô tả nó như ví dụ kỹ thuật.

In [ ]:
label_counts = count_labels(frame, LABEL_COLUMN)
print(label_counts.to_string(index=False))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
label_figure = plot_label_distribution(label_counts, OUTPUT_DIR / "label_distribution.png")
print(f"Da luu bieu do: {label_figure}")
plt.figure(figsize=(8, 5))
plt.imshow(plt.imread(label_figure))
plt.axis("off")
if not os.environ.get("VIHSD_NOTEBOOK_SUPPRESS_DISPLAY"):
    plt.show()

## 2. Thống kê độ dài bình luận

`whitespace_token_count` là số đơn vị tách theo khoảng trắng, chưa phải số từ đã tách bằng bộ tách từ tiếng Việt. So sánh trung vị/khoảng giá trị giữa các nhãn sau khi có dữ liệu thật.

In [ ]:
lengths = comment_lengths(frame, TEXT_COLUMN, LABEL_COLUMN)
summary = length_summary(lengths)
print(summary.to_string(index=False))
length_figure = plot_comment_lengths(lengths, OUTPUT_DIR / "comment_lengths.png")
print(f"Da luu bieu do: {length_figure}")
plt.figure(figsize=(9, 5))
plt.imshow(plt.imread(length_figure))
plt.axis("off")
if not os.environ.get("VIHSD_NOTEBOOK_SUPPRESS_DISPLAY"):
    plt.show()

## 3. Kiểm tra dữ liệu trùng và nhãn

Bảng dưới đây chỉ có số đếm, tỷ lệ và mẫu minh họa đã lược bỏ nội dung. Giải thích vì sao bình luận trùng khác nhãn cần được kiểm tra thay vì tự sửa nhãn.

In [ ]:
allowed_labels = ["CLEAN", "OFFENSIVE", "HATE"]
issues = quality_issue_table(frame, TEXT_COLUMN, LABEL_COLUMN, allowed_labels)
print(issues.to_string(index=False))

## 4. Sửa một quy tắc và chạy lại tiền xử lý

Thử thêm `tks → cảm ơn` vào bản sao của từ điển. Quy tắc chỉ áp dụng cho token độc lập. Xem phần so sánh văn bản ở bộ nhớ notebook; không lưu output này nếu đang dùng bình luận thật. Phủ định, dấu câu và emoji cần được giữ.

In [ ]:
custom_rules = DEFAULT_ABBREVIATIONS.copy()
custom_rules["tks"] = "cảm ơn"
original_texts = frame[TEXT_COLUMN].copy()
processed_texts = normalize_series(original_texts, custom_rules)
change_summary = preprocessing_summary(original_texts, processed_texts)
print(change_summary)
if USE_SAMPLE_FIXTURE or SHOW_TEXT_PREVIEW:
    preview = compare_text_versions(original_texts, custom_rules).head(6)
    print(preview.to_string(index=False))
else:
    print("Noi dung mau duoc an. Neu duoc phep, bat SHOW_TEXT_PREVIEW chi de xem cuc bo.")

## 5. Câu hỏi tự trả lời

1. Nhãn nào có nhiều mẫu nhất? Hãy nêu số lượng và tỷ lệ từ bảng vừa tạo.
2. Độ dài trung vị có khác nhau giữa các nhãn không? Trích dẫn đúng số đo.
3. Có bao nhiêu dòng trùng theo quy tắc NFC/khoảng trắng/không phân biệt hoa thường? Vì sao chưa xóa tự động?
4. Quy tắc `tks` làm thay đổi bao nhiêu dòng? Từ `không`, dấu câu và emoji có còn không?
5. Một cách viết bất thường có thể giúp phân biệt nhãn nhưng cũng làm mô hình dễ học thuộc như thế nào?

**Gợi ý:** Không ghi nhận xét định lượng về ViHSD khi `SOURCE_MODE` là fixture. Với dữ liệu thật, lưu lệnh chạy và kiểm tra nguồn trước khi đưa số liệu vào báo cáo.